# Testing on Whole class

# Landmark based (noraml, copying , gesture )

In [ ]:
import cv2
import numpy as np

!pip install ultralytics
from tqdm import tqdm
from ultralytics import YOLO
from collections import defaultdict, deque


# Load models
tracker = YOLO("yolov8n.pt")        # YOLO for person detection/tracking
pose_model = YOLO("yolov8n-pose.pt")  # YOLO Pose for landmarks

# Video paths
VIDEO_PATH = "/content/drive/MyDrive/ourDataset/Testing samples/0129(c).MOV"
OUTPUT_PATH = "/content/drive/MyDrive/FYP/annotated/video_0129_c.mp4"

# Settings
SCORE_THRESHOLD = 6
CLIP_MEMORY = 20

# State
tracker_to_student = {}
student_id = 0
copy_scores = defaultdict(float)
history = defaultdict(lambda: deque(maxlen=CLIP_MEMORY))

# Rule functions (using COCO pose keypoints from YOLOv8 pose)
def side_glance(kp):
    nose, leye, reye, lear, rear = kp[0], kp[1], kp[2], kp[3], kp[4]
    # If one ear is not visible, or nose is far from face center, flag
    if lear[2] < 0.3 or rear[2] < 0.3:
        return 1
    face_center = (leye[0] + reye[0]) / 2
    return 1 if abs(nose[0] - face_center) > 25 else 0

def leaning(kp):
    ls, rs = kp[5], kp[6]  # left and right shoulders
    return 1 if abs(ls[1] - rs[1]) > 35 else 0

def reaching(kp):
    lw, rw = kp[9], kp[10]  # left and right wrists
    torso = (kp[5][:2] + kp[6][:2]) / 2
    if np.linalg.norm(lw[:2] - torso) > 150 or np.linalg.norm(rw[:2] - torso) > 150:
        return 1
    return 0

def hand_raise(kp):
    lw, rw = kp[9], kp[10]
    ls, rs = kp[5], kp[6]
    # Check if wrist is substantially above shoulder
    return (lw[1] < ls[1] - 20) or (rw[1] < rs[1] - 20)

# Draw pose skeleton and dashboard (optional)
def draw_pose(frame, kp, offset):
    ox, oy = offset
    SKELETON = [(0,1),(0,2),(1,3),(2,4),(5,6),(5,9),(6,10),(5,11),(6,12),(11,12)]
    for i,j in SKELETON:
        if kp[i][2]>0.3 and kp[j][2]>0.3:
            x1,y1 = int(kp[i][0])+ox, int(kp[i][1])+oy
            x2,y2 = int(kp[j][0])+ox, int(kp[j][1])+oy
            cv2.line(frame,(x1,y1),(x2,y2),(255,0,0),2)
    for i,(x,y,c) in enumerate(kp):
        if i not in [7,8] and c>0.3:
            cv2.circle(frame, (int(x)+ox,int(y)+oy), 4, (0,255,255), -1)

cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS) or 25
width = int(cap.get(3)); height = int(cap.get(4))
out = cv2.VideoWriter(OUTPUT_PATH, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Detect and track persons
    results = tracker.track(frame, persist=True, classes=[0], conf=0.4, verbose=False)

    # Check if there are any tracked person IDs before proceeding
    if results[0].boxes.id is None or len(results[0].boxes.id) == 0:
        out.write(frame)
        continue

    boxes = results[0].boxes.xyxy.cpu().numpy()
    ids = results[0].boxes.id.cpu().numpy().astype(int)

    for box, tid in zip(boxes, ids):
        x1,y1,x2,y2 = map(int, box)
        crop = frame[y1:y2, x1:x2]
        if crop.size == 0:
            continue

        # Assign a stable student ID
        if tid not in tracker_to_student:
            student_id += 1
            tracker_to_student[tid] = student_id
        sid = tracker_to_student[tid]

        # Pose landmarks
        pose_res = pose_model(crop, verbose=False)
        if len(pose_res[0].keypoints.data) == 0:
            continue
        kp = pose_res[0].keypoints.data[0].cpu().numpy()
        prev_kp = history[sid][-1] if history[sid] else None

        # Update copying scores
        g = side_glance(kp); l = leaning(kp); r = reaching(kp)
        if g:  copy_scores[sid] += 2
        if l:  copy_scores[sid] += 1
        if r:  copy_scores[sid] += 2
        # Cap score and apply decay
        copy_scores[sid] = min(copy_scores[sid], 10)
        if not (g or l or r):
            copy_scores[sid] *= 0.85

        history[sid].append(kp)

        # Detect gesture (raised hand)
        gesture = False
        if hand_raise(kp):
            # Require persistence: check previous frame too if available
            if prev_kp is None or hand_raise(prev_kp):
                gesture = True

        # Determine label and color
        label = "normal"; conf = 0.0; color = (0,255,0)
        if gesture:
            label = "cheating_gesture"; conf = 0.90; color = (0,0,255)
        elif copy_scores[sid] > SCORE_THRESHOLD:
            label = "cheating_copying"
            conf = min(copy_scores[sid]/10, 0.95); color = (0,165,255)

        # Draw box and label
        cv2.rectangle(frame, (x1,y1),(x2,y2), color, 3)
        cv2.putText(frame, f"ID:{sid} {label} {conf:.2f}", (x1,y1-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
        draw_pose(frame, kp, (x1,y1))

    out.write(frame)

cap.release()
out.release()
print("✅ Processing complete. Output saved to", OUTPUT_PATH)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 24.4 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 2 packages in 329ms
Prepared 1 package in 55ms
Installed 1 package in 4ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 0.8s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

✅ Processing complete. Output saved to /content/drive/MyDrive/FYP/annotated/video_0129_c.mp4


# Fusion with videomae

In [ ]:
# =========================================================
# IMPROVED CHEATING DETECTION SYSTEM
# STRICT TEMPORAL RULE-BASED + VIDEOMAE FUSION
# =========================================================

!pip install -q ultralytics transformers accelerate opencv-python scikit-learn

# =========================================================
# IMPORTS
# =========================================================

import cv2
import torch
import numpy as np
import pandas as pd

from ultralytics import YOLO
from collections import defaultdict, deque

from transformers import (
    VideoMAEImageProcessor,
    VideoMAEForVideoClassification
)

# =========================================================
# DEVICE
# =========================================================

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

# =========================================================
# PATHS
# =========================================================

VIDEO_PATH = "/content/drive/MyDrive/ourDataset/Testing samples/0129(c).MOV"

VIDEOMAE_MODEL = "/content/drive/MyDrive/FYP/Models/final_model.pth"

OUTPUT_VIDEO = "/content/drive/MyDrive/FYP/annotated/final_output.mp4"

CSV_OUTPUT = "/content/drive/MyDrive/FYP/annotated/final_logs.csv"

# =========================================================
# CLASSES
# =========================================================

class_names = [
    "cheating_copying",
    "cheating_gesture",
    "normal"
]

# =========================================================
# MODELS
# =========================================================

tracker = YOLO("yolov8n.pt")

pose_model = YOLO("yolov8n-pose.pt")

# =========================================================
# VIDEOMAE
# =========================================================

processor = VideoMAEImageProcessor.from_pretrained(
    "MCG-NJU/videomae-base-finetuned-kinetics"
)

videomae = VideoMAEForVideoClassification.from_pretrained(
    "MCG-NJU/videomae-base-finetuned-kinetics",
    num_labels=3,
    ignore_mismatched_sizes=True
)

checkpoint = torch.load(
    VIDEOMAE_MODEL,
    map_location=device
)

if "model_state_dict" in checkpoint:
    checkpoint = checkpoint["model_state_dict"]

model_dict = videomae.state_dict()

filtered = {}

for k, v in checkpoint.items():

    if k in model_dict:

        if v.shape == model_dict[k].shape:

            filtered[k] = v

videomae.load_state_dict(
    filtered,
    strict=False
)

videomae.to(device)

videomae.eval()

print("✅ VideoMAE Loaded")

# =========================================================
# SETTINGS
# =========================================================

CLIP_LEN = 16

FRAME_SKIP = 2

# STRICT TEMPORAL THRESHOLDS

COPY_TEMPORAL_FRAMES = 12
GESTURE_TEMPORAL_FRAMES = 10

# =========================================================
# STORAGE
# =========================================================

student_buffers = defaultdict(list)

prediction_history = defaultdict(
    lambda: deque(maxlen=15)
)

tracker_map = {}

sid_counter = 0

# temporal counters

side_counter = defaultdict(int)

gesture_counter = defaultdict(int)

copy_counter = defaultdict(int)

# =========================================================
# VIDEOMAE PREDICTION
# =========================================================

def predict_videomae(frames):

    processed = []

    for f in frames[-CLIP_LEN:]:

        rgb = cv2.cvtColor(
            f,
            cv2.COLOR_BGR2RGB
        )

        rgb = cv2.resize(
            rgb,
            (224,224)
        )

        processed.append(rgb)

    inputs = processor(
        processed,
        return_tensors="pt"
    )

    pixel_values = inputs[
        "pixel_values"
    ].to(device)

    with torch.no_grad():

        outputs = videomae(
            pixel_values=pixel_values
        )

        probs = torch.softmax(
            outputs.logits,
            dim=1
        )

        pred = torch.argmax(
            probs,
            dim=1
        ).item()

        conf = probs[0][pred].item()

    return class_names[pred], conf

# =========================================================
# LANDMARK CONNECTIONS
# =========================================================

SKELETON = [

    (0,1),(0,2),
    (1,3),(2,4),

    (5,6),

    (5,7),(7,9),
    (6,8),(8,10),

    (5,11),(6,12),

    (11,12),

    (11,13),(13,15),
    (12,14),(14,16)
]

# =========================================================
# RULES
# =========================================================

def side_glance(kp):

    nose = kp[0]

    leye = kp[1]

    reye = kp[2]

    if leye[2] < 0.4 or reye[2] < 0.4:
        return False

    eye_center = (
        leye[0] + reye[0]
    ) / 2

    dist = abs(
        nose[0] - eye_center
    )

    return dist > 18

# =========================================================

def leaning(kp):

    ls = kp[5]

    rs = kp[6]

    diff = abs(
        ls[1] - rs[1]
    )

    return diff > 40

# =========================================================

def looking_down(kp):

    nose = kp[0]

    ls = kp[5]

    rs = kp[6]

    shoulder_y = (
        ls[1] + rs[1]
    ) / 2

    return nose[1] > shoulder_y - 20

# =========================================================
# STRICT GESTURE RULE
# =========================================================

def cheating_gesture(kp):

    lw = kp[9]
    rw = kp[10]

    ls = kp[5]
    rs = kp[6]

    le = kp[7]
    re = kp[8]

    # hand above shoulder

    left_up = lw[1] < ls[1] + 10
    right_up = rw[1] < rs[1] + 10

    # hand NOT near writing position

    left_far = abs(
        lw[0] - ls[0]
    ) > 40

    right_far = abs(
        rw[0] - rs[0]
    ) > 40

    # elbow bent

    left_arm = np.linalg.norm(
        lw[:2] - le[:2]
    )

    right_arm = np.linalg.norm(
        rw[:2] - re[:2]
    )

    left_valid = (
        left_up and
        left_far and
        left_arm > 45
    )

    right_valid = (
        right_up and
        right_far and
        right_arm > 45
    )

    return left_valid or right_valid

# =========================================================
# LOOKING OTHER STUDENT
# =========================================================

def looking_other_student(kp1, kp2):

    nose1 = kp1[0]

    eyes_center = (
        kp1[1][:2] +
        kp1[2][:2]
    ) / 2

    direction = nose1[:2] - eyes_center

    target = kp2[0][:2] - nose1[:2]

    dot = np.dot(
        direction,
        target
    )

    return dot > 0

# =========================================================
# VIDEO
# =========================================================

cap = cv2.VideoCapture(VIDEO_PATH)

fps = cap.get(cv2.CAP_PROP_FPS)

width = int(cap.get(3))

height = int(cap.get(4))

out = cv2.VideoWriter(
    OUTPUT_VIDEO,
    cv2.VideoWriter_fourcc(*'mp4v'),
    fps / FRAME_SKIP,
    (width,height)
)

# =========================================================
# LOGS
# =========================================================

logs = []

frame_no = 0

# =========================================================
# MAIN LOOP
# =========================================================

while True:

    ret, frame = cap.read()

    if not ret:
        break

    frame_no += 1

    if frame_no % FRAME_SKIP != 0:
        continue

    # =====================================================
    # DETECTION + TRACKING
    # =====================================================

    results = tracker.track(
        frame,
        persist=True,
        classes=[0],
        conf=0.45,
        verbose=False
    )

    if results[0].boxes.id is None:

        out.write(frame)

        continue

    boxes = results[0].boxes.xyxy.cpu().numpy()

    ids = results[0].boxes.id.cpu().numpy().astype(int)

    students = []

    # =====================================================
    # FIRST PASS
    # =====================================================

    for box, tid in zip(boxes, ids):

        x1,y1,x2,y2 = map(int, box)

        crop = frame[y1:y2, x1:x2]

        if crop.size == 0:
            continue

        if tid not in tracker_map:

            sid_counter += 1

            tracker_map[tid] = sid_counter

        sid = tracker_map[tid]

        # =================================================
        # BUFFER
        # =================================================

        student_buffers[sid].append(crop)

        if len(student_buffers[sid]) > CLIP_LEN:

            student_buffers[sid] = \
            student_buffers[sid][-CLIP_LEN:]

        # =================================================
        # POSE
        # =================================================

        pose = pose_model(
            crop,
            verbose=False
        )

        if len(
            pose[0].keypoints.data
        ) == 0:

            continue

        kp = pose[0].keypoints.data[
            0
        ].cpu().numpy()

        students.append({

            "sid": sid,
            "box": (x1,y1,x2,y2),
            "kp": kp,
            "crop": crop
        })

    # =====================================================
    # SECOND PASS
    # =====================================================

    for student in students:

        sid = student["sid"]

        x1,y1,x2,y2 = student["box"]

        kp = student["kp"]

        crop = student["crop"]

        # =================================================
        # RULES
        # =================================================

        side = side_glance(kp)

        lean = leaning(kp)

        gesture = cheating_gesture(kp)

        look_other = False

        for other in students:

            if other["sid"] == sid:
                continue

            if looking_other_student(
                kp,
                other["kp"]
            ):

                look_other = True
                break

        # =================================================
        # TEMPORAL COPYING
        # =================================================

        if side and look_other:

            side_counter[sid] += 1

        else:

            side_counter[sid] = max(
                0,
                side_counter[sid] - 1
            )

        # =================================================
        # STRICT COPYING
        # =================================================

        copying = (
            side_counter[sid]
            >= COPY_TEMPORAL_FRAMES
        )

        # =================================================
        # STRICT GESTURE
        # =================================================

        if gesture:

            gesture_counter[sid] += 1

        else:

            gesture_counter[sid] = max(
                0,
                gesture_counter[sid] - 1
            )

        gesture_detected = (
            gesture_counter[sid]
            >= GESTURE_TEMPORAL_FRAMES
        )

        # =================================================
        # VIDEOMAE
        # =================================================

        videomae_label = "normal"

        videomae_conf = 0.5

        if len(
            student_buffers[sid]
        ) >= CLIP_LEN:

            videomae_label, videomae_conf = \
            predict_videomae(
                student_buffers[sid]
            )

        # =================================================
        # FINAL FUSION
        # =================================================

        final_label = "normal"

        final_conf = 0.50

        # STRICT RULES FIRST

        if gesture_detected:

            final_label = "cheating_gesture"

            final_conf = max(
                0.70,
                videomae_conf
            )

        elif copying:

            final_label = "cheating_copying"

            final_conf = max(
                0.70,
                videomae_conf
            )

        # MODEL SUPPORT

        elif (
            videomae_label != "normal"
            and videomae_conf > 0.85
        ):

            final_label = videomae_label

            final_conf = videomae_conf

        # =================================================
        # TEMPORAL SMOOTHING
        # =================================================

        prediction_history[sid].append(
            final_label
        )

        stable_label = max(
            set(prediction_history[sid]),
            key=prediction_history[sid].count
        )

        # =================================================
        # DRAW SKELETON
        # =================================================

        for p1, p2 in SKELETON:

            if (
                kp[p1][2] > 0.3
                and
                kp[p2][2] > 0.3
            ):

                xA, yA = int(kp[p1][0]), int(kp[p1][1])

                xB, yB = int(kp[p2][0]), int(kp[p2][1])

                cv2.line(
                    crop,
                    (xA,yA),
                    (xB,yB),
                    (0,255,255),
                    2
                )

        # =================================================
        # DRAW POINTS
        # =================================================

        for point in kp:

            px, py, conf = point

            if conf > 0.3:

                cv2.circle(
                    crop,
                    (int(px), int(py)),
                    4,
                    (255,0,0),
                    -1
                )

        # =================================================
        # DRAW BOX
        # =================================================

        color = (0,255,0)

        if stable_label != "normal":

            color = (0,0,255)

        cv2.rectangle(
            frame,
            (x1,y1),
            (x2,y2),
            color,
            3
        )

        # =================================================
        # TEXT
        # =================================================

        text = (
            f"ID:{sid} "
            f"{stable_label} "
            f"{final_conf:.2f}"
        )

        cv2.putText(
            frame,
            text,
            (x1,y1-10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            color,
            2
        )

        # =================================================
        # DEBUG
        # =================================================

        debug = (
            f"S:{side_counter[sid]} "
            f"G:{gesture_counter[sid]}"
        )

        cv2.putText(
            frame,
            debug,
            (x1,y2+20),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (255,255,0),
            2
        )

        # =================================================
        # LOGS
        # =================================================

        logs.append({

            "frame": frame_no,

            "student_id": sid,

            "videomae_label": videomae_label,

            "videomae_conf": videomae_conf,

            "copy_counter": side_counter[sid],

            "gesture_counter": gesture_counter[sid],

            "final_label": stable_label,

            "final_conf": final_conf
        })

    out.write(frame)

# =========================================================
# SAVE
# =========================================================

cap.release()

out.release()

df = pd.DataFrame(logs)

df.to_csv(
    CSV_OUTPUT,
    index=False
)

print("\n✅ DONE")

print("Video Saved:", OUTPUT_VIDEO)

print("CSV Saved:", CSV_OUTPUT)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
